In [2]:
import os
import joblib
import pandas as pd

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_train = joblib.load("/content/processed/X_train.pkl")
X_test = joblib.load("/content/processed/X_test.pkl")
y_train = joblib.load("/content/processed/y_train.pkl")
y_test = joblib.load("/content/processed/y_test.pkl")

print("Đã nạp dữ liệu thành công!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/processed/X_train.pkl'

In [3]:
import joblib

X_train = joblib.load("/content/processed/X_train.pkl")
X_test = joblib.load("/content/processed/X_test.pkl")
y_train = joblib.load("/content/processed/y_train.pkl")
y_test = joblib.load("/content/processed/y_test.pkl")

print("Đã nạp dữ liệu thành công!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/processed/X_train.pkl'

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Đọc dataset
DATA_PATH = "/content/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

# Chuyển TotalCharges sang dạng số
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Chuyển nhãn Churn thành 0 và 1
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Tách X và y
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Xác định loại cột
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

# Tiền xử lý cột số
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Tiền xử lý cột chữ
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Kết hợp tiền xử lý
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Chuyển đổi dữ liệu
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Đã chuẩn bị dữ liệu!")
print("X_train:", X_train_processed.shape)
print("X_test:", X_test_processed.shape)

Đã chuẩn bị dữ liệu!
X_train: (5634, 45)
X_test: (1409, 45)


In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
}

trained_models = {}

for name, model in models.items():
    print(f"Đang huấn luyện {name}...")
    model.fit(X_train_processed, y_train)
    trained_models[name] = model
    print(f"{name} hoàn thành!")

Đang huấn luyện KNN...
KNN hoàn thành!
Đang huấn luyện Decision Tree...
Decision Tree hoàn thành!
Đang huấn luyện Logistic Regression...
Logistic Regression hoàn thành!
Đang huấn luyện Random Forest...
Random Forest hoàn thành!


In [6]:
import os
import joblib

os.makedirs("/content/ai-models", exist_ok=True)

for name, model in trained_models.items():
    file_name = name.lower().replace(" ", "_") + ".pkl"
    path = f"/content/ai-models/{file_name}"

    joblib.dump(model, path)
    print("Đã lưu:", path)

joblib.dump(preprocessor, "/content/ai-models/preprocessor.pkl")

print("Hoàn thành phần 03 - Train!")

Đã lưu: /content/ai-models/knn.pkl
Đã lưu: /content/ai-models/decision_tree.pkl
Đã lưu: /content/ai-models/logistic_regression.pkl
Đã lưu: /content/ai-models/random_forest.pkl
Hoàn thành phần 03 - Train!
